In [1]:
import sys
print("python exe:", sys.executable)   # full path to the interpreter
print("python ver:", sys.version)

# try torch only in the working notebook
try:
    import torch
    print("torch      :", torch.__version__, torch.__file__)
except ModuleNotFoundError as e:
    print("torch not importable:", e)



python exe: c:\Users\aneek\anaconda3\envs\tf_gpu_env\python.exe
python ver: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]
torch      : 1.12.1+cu113 c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\torch\__init__.py


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import tensorflow as tf
print(tf.__version__)  # This should print the version of TensorFlow
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

print("CUDA version:", tf.sysconfig.get_build_info()["cuda_version"])
print("cuDNN version:", tf.sysconfig.get_build_info()["cudnn_version"])

from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

2.10.0
Num GPUs Available:  1
CUDA version: 64_112
cuDNN version: 64_8
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 3051009055860398589
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5713690624
locality {
  bus_id: 1
  links {
  }
}
incarnation: 3540272106533483433
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


In [3]:
import time
import tensorflow as tf
import psutil

class PowerMonitor:
    def __init__(self):
        self.gpu_available = tf.config.list_physical_devices('GPU')
        
        # Hardware power specifications (adjust these values for your system)
        self.cpu_tdp = 65    # Typical TDP for desktop CPUs in watts
        self.gpu_tdp = 250   # Typical TDP for desktop GPUs in watts
        
    def get_stats(self):
        """Get system stats with power estimation"""
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'ram_mb': psutil.virtual_memory().used / (1024**2),
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85  # Base CPU power
        }
        
        if self.gpu_available:
            try:
                # TensorFlow GPU memory monitoring
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                stats.update({
                    'gpu_mem_mb': mem_info['current'] / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75  # Add GPU power estimate
                })
            except:
                pass
                
        return stats

# Initialize monitor
monitor = PowerMonitor()

In [4]:
import time
import torch
import psutil
import os

class PowerMonitor1:
    def __init__(self):
        self.gpu_available = torch.cuda.is_available()
        self.process = psutil.Process(os.getpid())  # Track current process
        
        # Hardware power specifications
        self.cpu_tdp = 65
        self.gpu_tdp = 250
        
    def get_stats(self):
        """Get process-specific stats with power estimation"""
        process_memory = self.process.memory_info()
        
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'process_ram_mb': process_memory.rss / (1024**2),  # Only this process's RAM
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85
        }
        
        if self.gpu_available:
            try:
                gpu_memory_allocated = torch.cuda.memory_allocated()
                stats.update({
                    'gpu_mem_mb': gpu_memory_allocated / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75
                })
            except Exception as e:
                print(f"Error retrieving GPU memory: {e}")
                
        return stats

# Initialize monitor1
monitor1 = PowerMonitor1()

# Model

In [5]:
import os
import random
import numpy as np
import torch
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "google/vit-base-patch32-224-in21k"

INPUT_SIZE = 160
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# ============================================================
# LOAD PRETRAINED ViT-32 PROCESSOR AND MODEL
# ============================================================

processor = ViTImageProcessor.from_pretrained(
    MODEL_NAME,
    size={
        "height": INPUT_SIZE,
        "width": INPUT_SIZE
    }
)

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,

    id2label={
        0: "real",
        1: "fake"
    },

    label2id={
        "real": 0,
        "fake": 1
    },

    # Replace the original pretrained classification head
    # with a new two-class classification head.
    ignore_mismatched_sizes=True
)

# Full-model fine-tuning
model = model.to(device)

print("\nOutput classes:", model.config.num_labels)
print("Patch size:", model.config.patch_size)
print("Image size in original checkpoint:", model.config.image_size)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)
##############################################
#dataset class
class DeepfakeViTDataset(Dataset):
    """
    Dataset for OpenCV-loaded BGR images.

    Labels:
        0 = real
        1 = fake
    """

    def __init__(
        self,
        images,
        labels,
        processor
    ):
        self.images = images

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )

        self.processor = processor

        if len(self.images) != len(self.labels):
            raise ValueError(
                "The numbers of images and labels do not match."
            )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index]
        label = int(self.labels[index])

        if isinstance(image, np.ndarray):
            if image.ndim != 3 or image.shape[-1] != 3:
                raise ValueError(
                    f"Invalid image shape at index {index}: "
                    f"{image.shape}"
                )

            # OpenCV BGR -> RGB
            image_rgb = image[..., ::-1]

            image_rgb = np.ascontiguousarray(
                image_rgb,
                dtype=np.uint8
            )

            image = Image.fromarray(image_rgb)

        elif isinstance(image, str):
            image = Image.open(image).convert("RGB")

        elif isinstance(image, Image.Image):
            image = image.convert("RGB")

        else:
            raise TypeError(
                f"Unsupported image type at index {index}: "
                f"{type(image)}"
            )

        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = processed[
            "pixel_values"
        ].squeeze(0)

        return {
            "pixel_values": pixel_values,

            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }
    #optimizer once
optimizer = optim.Adam(model.parameters(),lr=LEARNING_RATE)
    #Validation function

def evaluate_vit(
    model,
    data_loader,
    device
):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.inference_mode():
        for batch in data_loader:
            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                labels=labels,

                # Required because the model receives 160 × 160
                # instead of the checkpoint's original 224 × 224.
                interpolate_pos_encoding=True
            )

            loss = outputs.loss
            logits = outputs.logits

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = labels.size(0)

            total_loss += (
                loss.item() * current_batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += current_batch_size

    average_loss = total_loss / total_samples

    accuracy = (
        correct_predictions / total_samples
    )

    return average_loss, accuracy
#training function
def train_vit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=10
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for batch in train_loader:
            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(
                pixel_values=pixel_values,
                labels=labels,
                interpolate_pos_encoding=True
            )

            loss = outputs.loss
            logits = outputs.logits

            loss.backward()
            optimizer.step()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = labels.size(0)

            running_loss += (
                loss.item() * current_batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += current_batch_size

        train_loss = (
            running_loss / total_samples
        )

        train_accuracy = (
            correct_predictions / total_samples
        )

        val_loss, val_accuracy = evaluate_vit(
            model,
            val_loader,
            device
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train accuracy: {train_accuracy:.4f} | "
            f"Validation loss: {val_loss:.4f} | "
            f"Validation accuracy: {val_accuracy:.4f}"
        )

    return history

Device: cuda


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch32-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Output classes: 2
Patch size: 32
Image size in original checkpoint: 224
Trainable parameters: 87456770
Total parameters: 87456770


In [6]:
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
    auc
)


def evaluate_vit_complete(
    model,
    test_loader,
    device
):
    model.eval()

    all_labels = []
    all_predictions = []
    all_fake_probabilities = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch["pixel_values"].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                labels=labels,
                interpolate_pos_encoding=True
            )

            logits = outputs.logits
            loss = outputs.loss

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            # Probability of class 1: fake
            fake_probabilities = probabilities[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = labels.size(0)

            total_loss += (
                loss.item() * current_batch_size
            )

            total_samples += current_batch_size

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_fake_probabilities.extend(
                fake_probabilities.cpu().numpy()
            )

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_predictions)
    y_prob = np.asarray(all_fake_probabilities)

    test_loss = total_loss / total_samples

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    average_precision = average_precision_score(
        y_true,
        y_prob
    )

    pr_precision, pr_recall, _ = precision_recall_curve(
        y_true,
        y_prob
    )

    pr_auc = auc(
        pr_recall,
        pr_precision
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0.0
    )

    # Equal Error Rate
    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )

    fnr = 1.0 - tpr

    eer_index = np.nanargmin(
        np.abs(fpr - fnr)
    )

    eer = (
        fpr[eer_index]
        + fnr[eer_index]
    ) / 2.0

    eer_threshold = thresholds[eer_index]

    results = {
        "test_loss": test_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1_score": f1,
        "mcc": mcc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "average_precision": average_precision,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate
    }

    print("\n" + "=" * 60)
    print("VIT-32 TEST RESULTS")
    print("=" * 60)

    for metric_name, metric_value in results.items():
        print(
            f"{metric_name:25s}: "
            f"{metric_value:.6f}"
        )

    print("\nConfusion matrix:")
    print(cm)

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Real",
                "Fake"
            ],
            digits=4,
            zero_division=0
        )
    )

    return {
        "metrics": results,
        "confusion_matrix": cm,
        "true_labels": y_true,
        "predictions": y_pred,
        "fake_probabilities": y_prob
    }

# Wild deepfake

In [22]:
import h5py
import numpy as np

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

with h5py.File(H5_PATH, "r") as h5f:
    # Load image arrays
    train_images = h5f["train_images"][:]
    train_labels = h5f["train_labels"][:]

    val_images = h5f["val_images"][:]
    val_labels = h5f["val_labels"][:]

    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

# Verify dataset sizes
print(f"Total train: {len(train_images)} images")
print(f"Total validation: {len(val_images)} images")
print(f"Total test: {len(test_images)} images")

print(f"Train labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test labels: {len(test_labels)}")

# Verify shapes and data types
print("\nArray information:")
print(f"Train images: {train_images.shape}, dtype={train_images.dtype}")
print(f"Validation images: {val_images.shape}, dtype={val_images.dtype}")
print(f"Test images: {test_images.shape}, dtype={test_images.dtype}")

print(f"Train labels: {train_labels.shape}, dtype={train_labels.dtype}")
print(f"Validation labels: {val_labels.shape}, dtype={val_labels.dtype}")
print(f"Test labels: {test_labels.shape}, dtype={test_labels.dtype}")

# Verify class distributions
print("\nClass distribution:")
print(
    f"Train: Real={np.sum(train_labels == 0)}, "
    f"Fake={np.sum(train_labels == 1)}"
)
print(
    f"Validation: Real={np.sum(val_labels == 0)}, "
    f"Fake={np.sum(val_labels == 1)}"
)
print(
    f"Test: Real={np.sum(test_labels == 0)}, "
    f"Fake={np.sum(test_labels == 1)}"
)

Total train: 36000 images
Total validation: 6000 images
Total test: 18000 images
Train labels: 36000
Validation labels: 6000
Test labels: 18000

Array information:
Train images: (36000, 160, 160, 3), dtype=uint8
Validation images: (6000, 160, 160, 3), dtype=uint8
Test images: (18000, 160, 160, 3), dtype=uint8
Train labels: (36000,), dtype=uint8
Validation labels: (6000,), dtype=uint8
Test labels: (18000,), dtype=uint8

Class distribution:
Train: Real=9000, Fake=27000
Validation: Real=1500, Fake=4500
Test: Real=4500, Fake=13500


In [8]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_images,
    train_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_images,
    val_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_images,
    test_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 2250
Validation batches: 375
Testing batches: 1125


In [9]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2949 | Train accuracy: 0.8698 | Validation loss: 0.2546 | Validation accuracy: 0.9040
Epoch 02/10 | Train loss: 0.1248 | Train accuracy: 0.9519 | Validation loss: 0.3161 | Validation accuracy: 0.9008
Epoch 03/10 | Train loss: 0.0671 | Train accuracy: 0.9756 | Validation loss: 0.2917 | Validation accuracy: 0.9138
Epoch 04/10 | Train loss: 0.0499 | Train accuracy: 0.9823 | Validation loss: 0.2905 | Validation accuracy: 0.9068
Epoch 05/10 | Train loss: 0.0429 | Train accuracy: 0.9841 | Validation loss: 0.3171 | Validation accuracy: 0.9088
Epoch 06/10 | Train loss: 0.0352 | Train accuracy: 0.9881 | Validation loss: 0.3061 | Validation accuracy: 0.9057
Epoch 07/10 | Train loss: 0.0297 | Train accuracy: 0.9900 | Validation loss: 0.3324 | Validation accuracy: 0.9110
Epoch 08/10 | Train loss: 0.0275 | Train accuracy: 0.9910 | Validation loss: 0.2698 | Validation accuracy: 0.9163
Epoch 09/10 | Train loss: 0.0248 | Train accuracy: 0.9917 | Validation loss: 0.3270 | Va

In [10]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [11]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)


VIT-32 TEST RESULTS

Test loss:                  0.931693
Accuracy:                   0.793489
Balanced accuracy:          0.787607
Precision:                  0.910303
Recall:                     0.803870
Specificity:                0.771444
F1-score:                   0.853619
ROC-AUC:                    0.853771
PR-AUC:                     0.937285
Average Precision:          0.937281
EER:                        0.212452
EER threshold:              0.500000
False-positive:             0.228656
False-negative:             0.196230

Confusion matrix
Class order: [Real, Fake]
[[3471 1029]
 [2649 10851]]

Classification Report
              precision    recall  f1-score   support

        Real     0.5672    0.7714    0.6537      4500
        Fake     0.9103    0.8038    0.8536     13500

    accuracy                         0.7934     18000
   macro avg     0.7388    0.7876    0.7537     18000
weighted avg     0.8246    0.7934    0.8036     18000



In [13]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
#print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 14.1%
Time Usage: 514.3 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [14]:
end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 17.2%
Time Usage: 514.4 s
GPU Memory Used: 1344.3 MB
Power Consumption: 93W


save the model

In [16]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_patch32_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-16 model saved:", SAVE_DIR)

ViT-16 model saved: D:\thesis\results\vit_base_patch32_160


load the model

In [7]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_patch32_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("ViT-16 model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

Device: cuda
Model directory exists: True
ViT-16 model loaded successfully
Number of classes: 2
Label mapping: {0: 'real', 1: 'fake'}
Total parameters: 87456770
Trainable parameters: 87456770


In [18]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_14736\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [19]:
# ============================================================
# PYTORCH ViT-16 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-16 profiling runs:")
display(results_df)

Device: cuda
Test images: 18000
Test batches: 1125

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 125.91 seconds | 6.9947 ms/image | 142.96 images/s

Starting ViT-16 resource run 2/5
Run 2: 121.86 seconds | 6.7699 ms/image | 147.71 images/s

Starting ViT-16 resource run 3/5
Run 3: 118.15 seconds | 6.5637 ms/image | 152.35 images/s

Starting ViT-16 resource run 4/5
Run 4: 122.58 seconds | 6.8102 ms/image | 146.84 images/s

Starting ViT-16 resource run 5/5
Run 5: 131.06 seconds | 7.2812 ms/image | 137.34 images/s

Individual ViT-16 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,125.905238,6.994735,142.964664,2.816376,6.215625,5154.652344,5155.377927,5160.128906,0.725583,5.476562,...,19.965187,20.0,35.950392,99,16.279347,33.176,0.570199,1,18000,0.781515
1,121.858346,6.769908,147.712492,2.875426,6.659375,5156.964844,5157.167009,5161.781250,0.202165,4.816406,...,19.945799,20.0,35.602529,100,15.946631,52.540,0.540473,2,18000,0.781515
2,118.145967,6.563665,152.353910,2.879888,7.615625,5157.261719,5157.412109,5160.066406,0.150391,2.804688,...,19.944238,20.0,37.310409,100,16.324503,45.702,0.536554,3,18000,0.781515
3,122.584099,6.810228,146.837968,2.845303,6.753125,5130.273438,5130.429586,5135.093750,0.156148,4.820312,...,19.946188,20.0,35.724664,99,16.173649,36.044,0.551502,4,18000,0.781515
4,131.061757,7.281209,137.339835,2.815373,6.271875,5130.386719,5130.927667,5135.789062,0.540948,5.402344,...,19.949622,20.0,30.785055,74,14.067668,36.144,0.512451,5,18000,0.781515


In [21]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-32 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-32 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,123.911081,4.855923,117.881656,129.940506
1,latency_ms_per_image,6.883949,0.269774,6.548981,7.218917
2,throughput_images_per_s,145.441774,5.627180,138.454707,152.428841
3,average_cpu_percent,2.846473,0.030941,2.808055,2.884891
4,peak_cpu_percent,6.703125,0.561405,6.006048,7.400202
5,average_ram_mb,5146.262860,14.249135,5128.570222,5163.955498
6,peak_ram_mb,5150.571875,13.831458,5133.397851,5167.745899
7,average_incremental_ram_mb,0.355047,0.262999,0.028491,0.681604
8,peak_incremental_ram_mb,4.664062,1.085140,3.316683,6.011442
9,average_gpu_memory_mb,6454.930676,0.008602,6454.919995,6454.941356



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 6.884 ± 0.270 (95% CI: 6.549–7.219)
peak_ram_mb: 5150.572 ± 13.831 (95% CI: 5133.398–5167.746)
peak_gpu_memory_mb: 6454.980 ± 0.000 (95% CI: 6454.980–6454.980)
average_gpu_utilization_percent: 35.075 ± 2.493 (95% CI: 31.979–38.171)
average_gpu_power_w: 15.758 ± 0.956 (95% CI: 14.571–16.946)


genralization

In [12]:
print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (Vit-32):")
test_dataset = DeepfakeViTDataset(test_celeb,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of wild deepfake dataset on Celeb-DF(V2) (Vit-32):

VIT-32 TEST RESULTS
test_loss                : 0.702064
accuracy                 : 0.471733
balanced_accuracy        : 0.514511
precision                : 0.909689
recall                   : 0.461596
specificity              : 0.567426
f1_score                 : 0.612431
mcc                      : 0.017140
roc_auc                  : 0.525532
pr_auc                   : 0.905496
average_precision        : 0.905547
eer                      : 0.482776
eer_threshold            : 0.496208
false_positive_rate      : 0.432574
false_negative_rate      : 0.538404

Confusion matrix:
[[ 324  247]
 [2902 2488]]

Classification report:
              precision    recall  f1-score   support

        Real     0.1004    0.5674    0.1707       571
        Fake     0.9097    0.4616    0.6124      5390

    accuracy                         0.4717      5961
   macro avg     0.5051    0.5145    0.3915      5961
weighted avg     0.8322    0.471

In [9]:
#dfc on wilddeepfake
print("\nTest results of wild deepfake dataset on DFC (Vit-32):")
test_dataset = DeepfakeViTDataset(test_hog,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of wild deepfake dataset on DFC (Vit-32):

VIT-32 TEST RESULTS
test_loss                : 1.863810
accuracy                 : 0.529000
balanced_accuracy        : 0.529000
precision                : 0.522692
recall                   : 0.668000
specificity              : 0.390000
f1_score                 : 0.586479
mcc                      : 0.060380
roc_auc                  : 0.522684
pr_auc                   : 0.512802
average_precision        : 0.513711
eer                      : 0.487667
eer_threshold            : 0.831081
false_positive_rate      : 0.610000
false_negative_rate      : 0.332000

Confusion matrix:
[[ 585  915]
 [ 498 1002]]

Classification report:
              precision    recall  f1-score   support

        Real     0.5402    0.3900    0.4530      1500
        Fake     0.5227    0.6680    0.5865      1500

    accuracy                         0.5290      3000
   macro avg     0.5314    0.5290    0.5197      3000
weighted avg     0.5314    0.5290    0.51

In [11]:
print("\nTest results of wild deepfake dataset on FF++ (Vit-32):")
#ff++ on wilddeepfake
test_dataset = DeepfakeViTDataset(test_ff,test_ff_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of wild deepfake dataset on FF++ (Vit-32):

VIT-32 TEST RESULTS
test_loss                : 1.939260
accuracy                 : 0.566465
balanced_accuracy        : 0.563687
precision                : 0.482051
recall                   : 0.547042
specificity              : 0.580332
f1_score                 : 0.512494
mcc                      : 0.125775
roc_auc                  : 0.576163
pr_auc                   : 0.470756
average_precision        : 0.471523
eer                      : 0.433539
eer_threshold            : 0.438305
false_positive_rate      : 0.419668
false_negative_rate      : 0.452958

Confusion matrix:
[[838 606]
 [467 564]]

Classification report:
              precision    recall  f1-score   support

        Real     0.6421    0.5803    0.6097      1444
        Fake     0.4821    0.5470    0.5125      1031

    accuracy                         0.5665      2475
   macro avg     0.5621    0.5637    0.5611      2475
weighted avg     0.5755    0.5665    0.5692 

# Celeb

In [24]:
import os
import cv2
import numpy as np

SAVE_ROOT = r'D:\thesis\celeb_processed'

def load_split(split_name, class_name):
    """Reload saved frames, grouped by video."""
    base = os.path.join(SAVE_ROOT, split_name, class_name)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        vid_dir = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(vid_dir, f))
                  for f in sorted(os.listdir(vid_dir))]
        if frames:
            nested.append(frames)
            ids.append(vid_id)
    return nested, ids

# Reload ALL six splits
print("Loading frames...")
real_train_final,  real_train_ids  = load_split('train', 'real')
synth_train_final, synth_train_ids = load_split('train', 'fake')
real_val_final,    real_val_ids    = load_split('val',   'real')
synth_val_final,   synth_val_ids   = load_split('val',   'fake')
real_test_final,   real_test_ids   = load_split('test',  'real')
synth_test_final,  synth_test_ids  = load_split('test',  'fake')

print("✅ All frames reloaded")
print("Train -> real videos:", len(real_train_final), " fake videos:", len(synth_train_final))
print("Val   -> real videos:", len(real_val_final),   " fake videos:", len(synth_val_final))
print("Test  -> real videos:", len(real_test_final),  " fake videos:", len(synth_test_final))
print("Example frame shape:", np.shape(real_train_final[0][0]))  # expect (160,160,3)
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Flatten video-grouped frames into one image array and create labels.

    real_videos: list of videos, where each video is a list of frames
    fake_videos: list of videos, where each video is a list of frames

    Returns
    -------
    images : NumPy array with shape (N, 160, 160, 3)
    labels : NumPy array with shape (N,)
             0 = real, 1 = fake
    """

    # Flatten frames from all real videos
    real_frames = [
        frame
        for video_frames in real_videos
        for frame in video_frames
        if frame is not None
    ]

    # Flatten frames from all fake videos
    fake_frames = [
        frame
        for video_frames in fake_videos
        for frame in video_frames
        if frame is not None
    ]

    if len(real_frames) == 0:
        raise ValueError("No real frames were found.")

    if len(fake_frames) == 0:
        raise ValueError("No fake frames were found.")

    # Convert to NumPy arrays
    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    # Combine images
    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    # Create labels
    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training set
train_celeb, train_labels = combine_split(
    real_train_final,
    synth_train_final
)

# Validation set
val_celeb, val_labels = combine_split(
    real_val_final,
    synth_val_final
)

# Testing set
test_celeb, test_labels = combine_split(
    real_test_final,
    synth_test_final
)
print("\nTRAIN")
print("Images:", train_celeb.shape)
print("Labels:", train_labels.shape)
print("Real:", np.sum(train_labels == 0))
print("Fake:", np.sum(train_labels == 1))

print("\nVALIDATION")
print("Images:", val_celeb.shape)
print("Labels:", val_labels.shape)
print("Real:", np.sum(val_labels == 0))
print("Fake:", np.sum(val_labels == 1))

print("\nTEST")
print("Images:", test_celeb.shape)
print("Labels:", test_labels.shape)
print("Real:", np.sum(test_labels == 0))
print("Fake:", np.sum(test_labels == 1))

print("\nData types")
print("Train images:", train_celeb.dtype)
print("Train labels:", train_labels.dtype)

Loading frames...
✅ All frames reloaded
Train -> real videos: 354  fake videos: 3383
Val   -> real videos: 59  fake videos: 563
Test  -> real videos: 177  fake videos: 1693
Example frame shape: (160, 160, 3)

TRAIN
Images: (11899, 160, 160, 3)
Labels: (11899,)
Real: 1142
Fake: 10757

VALIDATION
Images: (1969, 160, 160, 3)
Labels: (1969,)
Real: 182
Fake: 1787

TEST
Images: (5961, 160, 160, 3)
Labels: (5961,)
Real: 571
Fake: 5390

Data types
Train images: uint8
Train labels: uint8


In [8]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_celeb,
    train_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_celeb,
    val_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_celeb,
    test_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 744
Validation batches: 124
Testing batches: 373


In [9]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2951 | Train accuracy: 0.9030 | Validation loss: 0.2759 | Validation accuracy: 0.9076
Epoch 02/10 | Train loss: 0.2560 | Train accuracy: 0.9063 | Validation loss: 0.2936 | Validation accuracy: 0.8873
Epoch 03/10 | Train loss: 0.2152 | Train accuracy: 0.9173 | Validation loss: 0.3358 | Validation accuracy: 0.8969
Epoch 04/10 | Train loss: 0.1728 | Train accuracy: 0.9292 | Validation loss: 0.2871 | Validation accuracy: 0.8944
Epoch 05/10 | Train loss: 0.1369 | Train accuracy: 0.9458 | Validation loss: 0.3397 | Validation accuracy: 0.8893
Epoch 06/10 | Train loss: 0.1157 | Train accuracy: 0.9529 | Validation loss: 0.3154 | Validation accuracy: 0.9121
Epoch 07/10 | Train loss: 0.0924 | Train accuracy: 0.9665 | Validation loss: 0.3347 | Validation accuracy: 0.8847
Epoch 08/10 | Train loss: 0.0795 | Train accuracy: 0.9718 | Validation loss: 0.3765 | Validation accuracy: 0.9005
Epoch 09/10 | Train loss: 0.0830 | Train accuracy: 0.9712 | Validation loss: 0.3984 | Va

In [10]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [11]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)


VIT-32 TEST RESULTS
test_loss                : 0.348812
accuracy                 : 0.927026
balanced_accuracy        : 0.629267
precision                : 0.927229
recall                   : 0.997588
specificity              : 0.260946
f1_score                 : 0.961123
mcc                      : 0.467946
roc_auc                  : 0.937976
pr_auc                   : 0.992256
average_precision        : 0.992247
eer                      : 0.141893
eer_threshold            : 0.999371
false_positive_rate      : 0.739054
false_negative_rate      : 0.002412

Confusion matrix:
[[ 149  422]
 [  13 5377]]

Classification report:
              precision    recall  f1-score   support

        Real     0.9198    0.2609    0.4065       571
        Fake     0.9272    0.9976    0.9611      5390

    accuracy                         0.9270      5961
   macro avg     0.9235    0.6293    0.6838      5961
weighted avg     0.9265    0.9270    0.9080      5961



In [13]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 9.3%
Time Usage: 60.1 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 11.1%
Time Usage: 62.3 s
GPU Memory Used: 1344.3 MB
Power Consumption: 93W


save the model

In [19]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_celeb_patch32_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-32 celeb model saved:", SAVE_DIR)

ViT-32 celeb model saved: D:\thesis\results\vit_base_celeb_patch32_160


load the model

In [ ]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_celeb_patch32_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("ViT-16 model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

Device: cuda
Model directory exists: True
ViT-16 model loaded successfully
Number of classes: 2
Label mapping: {0: 'real', 1: 'fake'}
Total parameters: 85800194
Trainable parameters: 85800194


In [16]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_42288\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [17]:
# ============================================================
# PYTORCH ViT-32 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-32 profiling runs:")
display(results_df)

Device: cuda
Test images: 5961
Test batches: 373

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 41.36 seconds | 6.9386 ms/image | 144.12 images/s

Starting ViT-16 resource run 2/5
Run 2: 39.96 seconds | 6.7035 ms/image | 149.17 images/s

Starting ViT-16 resource run 3/5
Run 3: 43.61 seconds | 7.3151 ms/image | 136.70 images/s

Starting ViT-16 resource run 4/5
Run 4: 41.42 seconds | 6.9479 ms/image | 143.93 images/s

Starting ViT-16 resource run 5/5
Run 5: 40.49 seconds | 6.7924 ms/image | 147.22 images/s

Individual ViT-32 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,41.360827,6.938572,144.121878,2.872139,6.215625,7493.292969,7494.200486,7498.562500,0.907518,5.269531,...,19.860334,24.019531,34.456464,77,16.059697,36.566,0.185262,1,5961,-3.556784
1,39.959802,6.703540,149.174914,2.892947,6.718750,7494.867188,7495.141554,7499.746094,0.274366,4.878906,...,19.857923,24.000000,36.439891,100,16.527150,30.754,0.184015,2,5961,-3.556784
2,43.605423,7.315119,136.703181,2.822924,5.825000,7495.269531,7495.403910,7500.066406,0.134379,4.796875,...,19.869674,24.000000,31.160401,100,14.097261,41.261,0.171210,3,5961,-3.556784
3,41.416168,6.947856,143.929297,2.880905,6.215625,7495.281250,7496.344912,7501.007812,1.063662,5.726562,...,19.852632,24.000000,34.515789,95,15.499850,31.076,0.178995,4,5961,-3.556784
4,40.489575,6.792413,147.223083,2.864496,6.271875,7496.222656,7408.674223,7501.031250,0.000000,4.808594,...,19.913747,24.000000,35.765499,100,16.146488,38.025,0.182399,5,5961,-3.556784


In [18]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,41.366359,1.393114,39.636579,43.096139
1,latency_ms_per_image,6.939500,0.233705,6.649317,7.229683
2,throughput_images_per_s,144.230470,4.747785,138.335317,150.125624
3,average_cpu_percent,2.866682,0.026653,2.833589,2.899776
4,peak_cpu_percent,6.249375,0.317487,5.855163,6.643587
5,average_ram_mb,7477.953017,38.735555,7429.856473,7526.049561
6,peak_ram_mb,7500.082812,1.022272,7498.813495,7501.352130
7,average_incremental_ram_mb,0.475985,0.478406,-0.118035,1.070005
8,peak_incremental_ram_mb,5.096094,0.402160,4.596746,5.595441
9,average_gpu_memory_mb,3408.038831,0.028140,3408.003890,3408.073772



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 6.939 ± 0.234 (95% CI: 6.649–7.230)
peak_ram_mb: 7500.083 ± 1.022 (95% CI: 7498.813–7501.352)
peak_gpu_memory_mb: 3412.172 ± 0.000 (95% CI: 3412.172–3412.172)
average_gpu_utilization_percent: 34.468 ± 2.032 (95% CI: 31.945–36.991)
average_gpu_power_w: 15.666 ± 0.951 (95% CI: 14.486–16.847)


#genralization

In [28]:
#wild deepfake on celeb
print("\nTest results of Celeb-DF(V2) on wild deepfake dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_images,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)




Test results of Celeb-DF(V2) on wild deepfake dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 1.043408
accuracy                 : 0.550056
balanced_accuracy        : 0.592333
precision                : 0.825009
recall                   : 0.507778
specificity              : 0.676889
f1_score                 : 0.628640
mcc                      : 0.160399
roc_auc                  : 0.616335
pr_auc                   : 0.833725
average_precision        : 0.833738
eer                      : 0.408407
eer_threshold            : 0.346583
false_positive_rate      : 0.323111
false_negative_rate      : 0.492222

Confusion matrix:
[[3046 1454]
 [6645 6855]]

Classification report:
              precision    recall  f1-score   support

        Real     0.3143    0.6769    0.4293      4500
        Fake     0.8250    0.5078    0.6286     13500

    accuracy                         0.5501     18000
   macro avg     0.5697    0.5923    0.5290     18000
weighted avg     0.6973    0.550

In [26]:
#DFC on celeb
print("\nTest results of Celeb-DF(V2) on DFC dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_hog,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of Celeb-DF(V2) on DFC dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 1.621302
accuracy                 : 0.468000
balanced_accuracy        : 0.468000
precision                : 0.480149
recall                   : 0.774000
specificity              : 0.162000
f1_score                 : 0.592649
mcc                      : -0.080925
roc_auc                  : 0.435938
pr_auc                   : 0.468566
average_precision        : 0.469204
eer                      : 0.560333
eer_threshold            : 0.869169
false_positive_rate      : 0.838000
false_negative_rate      : 0.226000

Confusion matrix:
[[ 243 1257]
 [ 339 1161]]

Classification report:
              precision    recall  f1-score   support

        Real     0.4175    0.1620    0.2334      1500
        Fake     0.4801    0.7740    0.5926      1500

    accuracy                         0.4680      3000
   macro avg     0.4488    0.4680    0.4130      3000
weighted avg     0.4488    0.4680    0.41

In [24]:
#FF++ on celeb
print("\nTest results of Celeb-DF(V2) on FF++ dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_ff,test_ff_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of Celeb-DF(V2) on FF++ dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 1.535369
accuracy                 : 0.559596
balanced_accuracy        : 0.599690
precision                : 0.483529
recall                   : 0.839961
specificity              : 0.359418
f1_score                 : 0.613749
mcc                      : 0.219795
roc_auc                  : 0.708487
pr_auc                   : 0.616261
average_precision        : 0.616739
eer                      : 0.324375
eer_threshold            : 0.922353
false_positive_rate      : 0.640582
false_negative_rate      : 0.160039

Confusion matrix:
[[519 925]
 [165 866]]

Classification report:
              precision    recall  f1-score   support

        Real     0.7588    0.3594    0.4878      1444
        Fake     0.4835    0.8400    0.6137      1031

    accuracy                         0.5596      2475
   macro avg     0.6212    0.5997    0.5508      2475
weighted avg     0.6441    0.5596    0.5403  

# DFC

In [26]:
import h5py
import numpy as np
# Open the HDF5 file in read mode
with h5py.File('D://thesis//dataset//deepfake dataset//resized_images.h5', 'r') as h5f:
    # Access each dataset
    celeb = np.array(h5f['celeb'])
    ffhq = np.array(h5f['ffhq'])
    gdwct = np.array(h5f['gdwct'])
    attgan = np.array(h5f['attgan'])
    stargan = np.array(h5f['stargan'])
    stylegan2 = np.array(h5f['stylegan2'])
    stylegan = np.array(h5f['stylegan'])

# Now, 'celeb', 'ffhq', etc., are NumPy arrays containing your datasets
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"ffhq shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"ffhq shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"ffhq shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"ffhq shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"ffhq shape: {stylegan.shape}, dtype: {stylegan.dtype}")
# Repeat for other datasets as needed
import cv2
# Function to resize images from (224, 224) to (160, 160)
def resize_images(image_array, target_size=(160, 160)):
    resized_images = np.array([cv2.resize(img, target_size) for img in image_array])
    return resized_images

celeb = resize_images(celeb, target_size=(160, 160))
ffhq = resize_images(ffhq, target_size=(160, 160))
gdwct = resize_images(gdwct, target_size=(160, 160))
attgan = resize_images(attgan, target_size=(160, 160))
stargan = resize_images(stargan, target_size=(160, 160))
stylegan = resize_images(stylegan, target_size=(160, 160))
stylegan2 = resize_images(stylegan2, target_size=(160, 160))
import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(celeb)), 2500)  # Get 2500 random indices
celeb = celeb[random_indices]  # Select the random subse

import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(ffhq)), 2500)  # Get 2500 random indices
ffhq = ffhq[random_indices]  # Select the random subse
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"gdwct shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"attagan shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"stargan shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"stylegan2 shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"stylegan shape: {stylegan.shape}, dtype: {stylegan.dtype}")
import random
import numpy as np

def split_data(data, train_ratio=0.7):
    """
    Splits data into training and testing sets based on the specified ratio.

    Parameters:
        data (list or np.array): The dataset to split.
        train_ratio (float): The ratio of the data to include in the training set.

    Returns:
        tuple: Two datasets - train and test.
    """
    # Shuffle the data
    random.shuffle(data)

    # Calculate the split index
    split_index = int(len(data) * train_ratio)

    # Split the data
    train_data = data[:split_index]
    test_data = data[split_index:]

    return train_data, test_data

# Split `celeb` into 70% train and 30% test
celeb_train_hog, celeb_test_hog = split_data(celeb, train_ratio=0.7)

# Split `ffhq` into 70% train and 30% test
ffhq_train_hog, ffhq_test_hog = split_data(ffhq, train_ratio=0.7)

# Split `attgan` into 70% train and 30% test
attgan_train_hog, attgan_test_hog = split_data(attgan, train_ratio=0.7)

# Split `stargan` into 70% train and 30% test
stargan_train_hog, stargan_test_hog = split_data(stargan, train_ratio=0.7)

# Split `gdwct` into 70% train and 30% test
gdwct_train_hog, gdwct_test_hog = split_data(gdwct, train_ratio=0.7)

# Split `stylegan2` into 70% train and 30% test_hog
stylegan2_train_hog, stylegan2_test_hog = split_data(stylegan2, train_ratio=0.7)

# Split `stylegan` into 70% train and 30% test_hog
stylegan_train_hog, stylegan_test_hog = split_data(stylegan, train_ratio=0.7)

# Convert to NumPy arrays if needed
celeb_train_hog, celeb_test_hog = np.array(celeb_train_hog), np.array(celeb_test_hog)
ffhq_train_hog, ffhq_test_hog = np.array(ffhq_train_hog), np.array(ffhq_test_hog)
attgan_train_hog, attgan_test_hog = np.array(attgan_train_hog), np.array(attgan_test_hog)
stargan_train_hog, stargan_test_hog = np.array(stargan_train_hog), np.array(stargan_test_hog)
gdwct_train_hog, gdwct_test_hog = np.array(gdwct_train_hog), np.array(gdwct_test_hog)
stylegan2_train_hog, stylegan2_test_hog = np.array(stylegan2_train_hog), np.array(stylegan2_test_hog)
stylegan_train_hog, stylegan_test_hog = np.array(stylegan_train_hog), np.array(stylegan_test_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_test: {len(celeb_test_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_test: {len(ffhq_test_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_test: {len(attgan_test_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_test: {len(stargan_test_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_test: {len(gdwct_test_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_test: {len(stylegan2_test_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_test: {len(stylegan_test_hog)} images")

########################################################################################################################################
#######################################divide into 60,10 train and val
#########################################################################################################################################
def extract_validation(train_data):
    """
    Extract every 10th sample from the training data and store it in a validation set.

    Parameters:
        train_data (list or np.array): The training dataset.

    Returns:
        tuple: Updated training dataset and validation dataset.
    """
    # Select every 10th sample for the validation set
    validation_data = train_data[::10]

    # Remove the selected samples from the training dataset
    updated_train_data = [train_data[i] for i in range(len(train_data)) if i % 10 != 0]

    return np.array(updated_train_data), np.array(validation_data)


# Perform the operation for each dataset
celeb_train_hog, celeb_val_hog = extract_validation(celeb_train_hog)
ffhq_train_hog, ffhq_val_hog = extract_validation(ffhq_train_hog)
attgan_train_hog, attgan_val_hog = extract_validation(attgan_train_hog)
stargan_train_hog, stargan_val_hog = extract_validation(stargan_train_hog)
gdwct_train_hog, gdwct_val_hog = extract_validation(gdwct_train_hog)
stylegan2_train_hog, stylegan2_val_hog = extract_validation(stylegan2_train_hog)
stylegan_train_hog, stylegan_val_hog = extract_validation(stylegan_train_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_val: {len(celeb_val_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_val: {len(ffhq_val_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_val: {len(attgan_val_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_val: {len(stargan_val_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_val: {len(gdwct_val_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_val: {len(stylegan2_val_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_val: {len(stylegan_val_hog)} images")
############################################################################################################################################################
#################################################concatenate the labels 0,1 real and fake
#############################################################################################################################################################


celeb_train_labels = np.zeros(len(celeb_train_hog), dtype=int)
ffhq_train_labels = np.zeros(len(ffhq_train_hog), dtype=int)
atta_train_labels = np.ones(len(attgan_train_hog), dtype=int)
star_train_labels = np.ones(len(stargan_train_hog), dtype=int)
gdwct_train_labels = np.ones(len(gdwct_train_hog), dtype=int)
stylegan2_train_labels = np.ones(len(stylegan2_train_hog), dtype=int)
stylegan_train_labels = np.ones(len(stylegan_train_hog), dtype=int)

# Concatenate all training datasets into a single `train` variable
train_hog = np.concatenate([celeb_train_hog, ffhq_train_hog, attgan_train_hog, stargan_train_hog, gdwct_train_hog, stylegan2_train_hog, stylegan_train_hog], axis=0)
train_labels=np.concatenate([celeb_train_labels, ffhq_train_labels, atta_train_labels, star_train_labels, gdwct_train_labels, stylegan2_train_labels,
                              stylegan_train_labels], axis=0)




celeb_test_labels = np.zeros(len(celeb_test_hog), dtype=int)
ffhq_test_labels = np.zeros(len(ffhq_test_hog), dtype=int)
atta_test_labels = np.ones(len(attgan_test_hog), dtype=int)
star_test_labels = np.ones(len(stargan_test_hog), dtype=int)
gdwct_test_labels = np.ones(len(gdwct_test_hog), dtype=int)
stylegan2_test_labels = np.ones(len(stylegan2_test_hog), dtype=int)
stylegan_test_labels = np.ones(len(stylegan_test_hog), dtype=int)

# Concatenate all testing datasets into a single `test` variable
test_hog = np.concatenate([celeb_test_hog, ffhq_test_hog, attgan_test_hog, stargan_test_hog, gdwct_test_hog, stylegan2_test_hog, stylegan_test_hog], axis=0)
test_labels = np.concatenate([celeb_test_labels, ffhq_test_labels, atta_test_labels, star_test_labels, gdwct_test_labels, stylegan2_test_labels,
                        stylegan_test_labels], axis=0)




celeb_val_labels = np.zeros(len(celeb_val_hog), dtype=int)
ffhq_val_labels = np.zeros(len(ffhq_val_hog), dtype=int)
atta_val_labels = np.ones(len(attgan_val_hog), dtype=int)
star_val_labels = np.ones(len(stargan_val_hog), dtype=int)
gdwct_val_labels = np.ones(len(gdwct_val_hog), dtype=int)
stylegan2_val_labels = np.ones(len(stylegan2_val_hog), dtype=int)
stylegan_val_labels = np.ones(len(stylegan_val_hog), dtype=int)

# Concatenate all validation datasets into a single `val` variable
val_hog = np.concatenate([celeb_val_hog, ffhq_val_hog, attgan_val_hog, stargan_val_hog, gdwct_val_hog, stylegan2_val_hog, stylegan_val_hog], axis=0)
val_labels = np.concatenate([celeb_val_labels, ffhq_val_labels, atta_val_labels, star_val_labels, gdwct_val_labels, stylegan2_val_labels,
                       stylegan_val_labels], axis=0)

# Print the results for verification
print(f"Total train: {len(train_hog)} images")
print(f"Total test: {len(test_hog)} images")
print(f"Total val: {len(val_hog)} images")


# Print results for verification
print(f"Train Labels: {len(train_labels)} ")
print(f"Test Labels: {len(test_labels)} ")
print(f"Val Labels: {len(val_labels)} ")



celeb shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
celeb shape: (2500, 160, 160, 3), dtype: uint8
ffhq shape: (2500, 160, 160, 3), dtype: uint8
gdwct shape: (1000, 160, 160, 3), dtype: uint8
attagan shape: (1000, 160, 160, 3), dtype: uint8
stargan shape: (1000, 160, 160, 3), dtype: uint8
stylegan2 shape: (1000, 160, 160, 3), dtype: uint8
stylegan shape: (1000, 160, 160, 3), dtype: uint8
celeb_train: 1750 images, celeb_test: 750 images
ffhq_train: 1750 images, ffhq_test: 750 images
attgan_train: 700 images, attgan_test: 300 images
stargan_train: 700 images, stargan_test: 300 images
gdwct_train: 700 images, gdwct_test: 300 images
stylegan2_train: 700 images, stylegan2_test: 300 images
stylegan_train: 700 images, stylegan

In [7]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_hog,
    train_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_hog,
    val_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_hog,
    test_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 394
Validation batches: 44
Testing batches: 188


In [8]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2482 | Train accuracy: 0.9000 | Validation loss: 0.1429 | Validation accuracy: 0.9457
Epoch 02/10 | Train loss: 0.0598 | Train accuracy: 0.9784 | Validation loss: 0.0990 | Validation accuracy: 0.9643
Epoch 03/10 | Train loss: 0.0552 | Train accuracy: 0.9832 | Validation loss: 0.0823 | Validation accuracy: 0.9714
Epoch 04/10 | Train loss: 0.0228 | Train accuracy: 0.9922 | Validation loss: 0.0951 | Validation accuracy: 0.9643
Epoch 05/10 | Train loss: 0.0318 | Train accuracy: 0.9886 | Validation loss: 0.0754 | Validation accuracy: 0.9829
Epoch 06/10 | Train loss: 0.0200 | Train accuracy: 0.9946 | Validation loss: 0.0963 | Validation accuracy: 0.9729
Epoch 07/10 | Train loss: 0.0198 | Train accuracy: 0.9937 | Validation loss: 0.0416 | Validation accuracy: 0.9886
Epoch 08/10 | Train loss: 0.0175 | Train accuracy: 0.9949 | Validation loss: 0.0655 | Validation accuracy: 0.9743
Epoch 09/10 | Train loss: 0.0196 | Train accuracy: 0.9933 | Validation loss: 0.0544 | Va

In [9]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [10]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)



VIT-32 TEST RESULTS
test_loss                     : 0.088562
accuracy                      : 0.977000
balanced_accuracy             : 0.977000
precision                     : 0.981818
recall_sensitivity            : 0.972000
specificity                   : 0.982000
f1_score                      : 0.976884
mcc                           : 0.954048
roc_auc                       : 0.996602
pr_auc                        : 0.997174
average_precision             : 0.997175
eer                           : 0.022667
eer_threshold                 : 0.398332
false_positive_rate           : 0.018000
false_negative_rate           : 0.028000
true_negatives                : 1473
false_positives               : 27
false_negatives               : 42
true_positives                : 1458
number_of_test_images         : 3000

Confusion Matrix:
[[1473   27]
 [  42 1458]]

Format:
[[TN, FP],
 [FN, TP]]

Classification Report:
              precision    recall  f1-score   support

        real     0.972277 

In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 8.8%
Time Usage: 2265.6 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [15]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 12.0%
Time Usage: 2267.7 s
GPU Memory Used: 1344.3 MB
Power Consumption: 93W


save the model

In [16]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_hog_patch32_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-32 DFC model saved:", SAVE_DIR)

ViT-32 DFC model saved: D:\thesis\results\vit_base_hog_patch32_160


#load the model

In [ ]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_hog_patch32_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("ViT-32 model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

In [17]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_8888\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [18]:
# ============================================================
# PYTORCH ViT-32 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-32 profiling runs:")
display(results_df)

Device: cuda
Test images: 3000
Test batches: 188

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 20.54 seconds | 6.8474 ms/image | 146.04 images/s

Starting ViT-16 resource run 2/5
Run 2: 20.01 seconds | 6.6691 ms/image | 149.95 images/s

Starting ViT-16 resource run 3/5
Run 3: 20.02 seconds | 6.6734 ms/image | 149.85 images/s

Starting ViT-16 resource run 4/5
Run 4: 21.19 seconds | 7.0626 ms/image | 141.59 images/s

Starting ViT-16 resource run 5/5
Run 5: 21.61 seconds | 7.2019 ms/image | 138.85 images/s

Individual ViT-32 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,20.542308,6.847436,146.040068,2.773602,5.375000,6793.582031,6793.829359,6798.460938,0.247327,4.878906,...,19.726316,24.0,35.463158,100,15.878505,28.590,0.091291,1,3000,-3.677584
1,20.007259,6.669086,149.945579,2.882855,6.718750,6794.273438,6794.425486,6795.589844,0.152048,1.316406,...,19.718919,24.0,36.470270,100,15.665551,33.091,0.087681,2,3000,-3.677584
2,20.020273,6.673424,149.848106,2.732592,5.771875,6794.871094,6794.990914,6798.515625,0.119820,3.644531,...,19.695652,24.0,35.277174,100,16.075196,29.047,0.090196,3,3000,-3.677584
3,21.187819,7.062606,141.590791,2.794427,5.375000,6794.878906,6795.025753,6799.671875,0.146847,4.792969,...,19.711340,24.0,30.680412,100,14.944371,60.391,0.088638,4,3000,-3.677584
4,21.605615,7.201872,138.852794,2.809628,5.375000,6794.886719,6795.038609,6799.683594,0.151890,4.796875,...,19.818182,24.0,30.777778,78,14.346783,39.412,0.086587,5,3000,-3.677584


In [21]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-32 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-32 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,20.672655,0.710836,19.790035,21.555274
1,latency_ms_per_image,6.890885,0.236945,6.596678,7.185091
2,throughput_images_per_s,145.255468,4.953008,139.105496,151.405439
3,average_cpu_percent,2.798620,0.055272,2.729992,2.867249
4,peak_cpu_percent,5.723125,0.582499,4.999857,6.446393
5,average_ram_mb,6794.662024,0.531875,6794.001613,6795.322434
6,peak_ram_mb,6798.384375,1.671685,6796.308704,6800.460046
7,average_incremental_ram_mb,0.163586,0.048678,0.103145,0.224028
8,peak_incremental_ram_mb,3.885937,1.524735,1.992729,5.779146
9,average_gpu_memory_mb,3408.530957,0.048366,3408.470903,3408.591011



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 6.891 ± 0.237 (95% CI: 6.597–7.185)
peak_ram_mb: 6798.384 ± 1.672 (95% CI: 6796.309–6800.460)
peak_gpu_memory_mb: 3412.797 ± 0.000 (95% CI: 3412.797–3412.797)
average_gpu_utilization_percent: 33.734 ± 2.780 (95% CI: 30.281–37.186)
average_gpu_power_w: 15.382 ± 0.719 (95% CI: 14.489–16.275)


#genralization

In [23]:
#wild deepfake on dfc
print("\nTest results of DFC on wild deepfake dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_images,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of DFC on wild deepfake dataset (ViT-16):

VIT-32 TEST RESULTS
test_loss                : 1.297549
accuracy                 : 0.602556
balanced_accuracy        : 0.488519
precision                : 0.744039
recall                   : 0.716593
specificity              : 0.260444
f1_score                 : 0.730058
mcc                      : -0.022202
roc_auc                  : 0.484122
pr_auc                   : 0.754041
average_precision        : 0.754091
eer                      : 0.520630
eer_threshold            : 0.903971
false_positive_rate      : 0.739556
false_negative_rate      : 0.283407

Confusion matrix:
[[1172 3328]
 [3826 9674]]

Classification report:
              precision    recall  f1-score   support

        Real     0.2345    0.2604    0.2468      4500
        Fake     0.7440    0.7166    0.7301     13500

    accuracy                         0.6026     18000
   macro avg     0.4893    0.4885    0.4884     18000
weighted avg     0.6167    0.6026    0.6

In [31]:
#celeb on dfc
print("\nTest results of DFC on Celeb-DF(V2) dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_celeb,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of DFC on Celeb-DF(V2) dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 0.808106
accuracy                 : 0.759604
balanced_accuracy        : 0.493629
precision                : 0.902871
recall                   : 0.822635
specificity              : 0.164623
f1_score                 : 0.860887
mcc                      : -0.009844
roc_auc                  : 0.471459
pr_auc                   : 0.892770
average_precision        : 0.892883
eer                      : 0.518749
eer_threshold            : 0.978778
false_positive_rate      : 0.835377
false_negative_rate      : 0.177365

Confusion matrix:
[[  94  477]
 [ 956 4434]]

Classification report:
              precision    recall  f1-score   support

        Real     0.0895    0.1646    0.1160       571
        Fake     0.9029    0.8226    0.8609      5390

    accuracy                         0.7596      5961
   macro avg     0.4962    0.4936    0.4884      5961
weighted avg     0.8250    0.7596    0.78

In [30]:
#FF++ on hog
print("\nTest results of dfc on FF++ dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_ff,test_ff_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of dfc on FF++ dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 3.064859
accuracy                 : 0.419394
balanced_accuracy        : 0.487860
precision                : 0.410097
recall                   : 0.898157
specificity              : 0.077562
f1_score                 : 0.563089
mcc                      : -0.042323
roc_auc                  : 0.457088
pr_auc                   : 0.382394
average_precision        : 0.383241
eer                      : 0.523655
eer_threshold            : 0.996837
false_positive_rate      : 0.922438
false_negative_rate      : 0.101843

Confusion matrix:
[[ 112 1332]
 [ 105  926]]

Classification report:
              precision    recall  f1-score   support

        Real     0.5161    0.0776    0.1349      1444
        Fake     0.4101    0.8982    0.5631      1031

    accuracy                         0.4194      2475
   macro avg     0.4631    0.4879    0.3490      2475
weighted avg     0.4720    0.4194    0.3132      

# FF++

LOAD THE DATASET

In [7]:
import os, cv2, numpy as np

FINAL_ROOT = r'D:\thesis\ff_final'

def load_split(split, cls):
    base = os.path.join(FINAL_ROOT, split, cls)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        d = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(d, f)) for f in sorted(os.listdir(d))]
        if frames:
            nested.append(frames); ids.append(vid_id)
    return nested, ids

# Main splits
ff_real_train_f, ff_real_train_ids = load_split('train', 'real')
ff_fake_train_f, ff_fake_train_ids = load_split('train', 'fake')
ff_real_val_f,   ff_real_val_ids   = load_split('val',   'real')
ff_fake_val_f,   ff_fake_val_ids   = load_split('val',   'fake')
ff_real_test_f,  ff_real_test_ids  = load_split('test',  'real')
ff_fake_test_f,  ff_fake_test_ids  = load_split('test',  'fake')

print("Reloaded main splits. Example shape:", np.shape(ff_real_train_f[0][0]))  # (160,160,3)
print("Real train videos:", len(ff_real_train_f), "| Fake train videos:", len(ff_fake_train_f))
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Combine all frames from the real and fake video groups.

    Labels:
        0 = Real
        1 = Fake
    """

    real_frames = [
        frame
        for video in real_videos
        for frame in video
        if frame is not None
    ]

    fake_frames = [
        frame
        for video in fake_videos
        for frame in video
        if frame is not None
    ]

    if not real_frames:
        raise ValueError("No real frames found.")

    if not fake_frames:
        raise ValueError("No fake frames found.")

    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training data
train_ff, train_ff_labels = combine_split(
    ff_real_train_f,
    ff_fake_train_f
)

# Validation data
val_ff, val_ff_labels = combine_split(
    ff_real_val_f,
    ff_fake_val_f
)

# Testing data
test_ff, test_ff_labels = combine_split(
    ff_real_test_f,
    ff_fake_test_f
)
print("\nTRAIN")
print("Images:", train_ff.shape)
print("Labels:", train_ff_labels.shape)
print("Real:", np.sum(train_ff_labels == 0))
print("Fake:", np.sum(train_ff_labels == 1))

print("\nVALIDATION")
print("Images:", val_ff.shape)
print("Labels:", val_ff_labels.shape)
print("Real:", np.sum(val_ff_labels == 0))
print("Fake:", np.sum(val_ff_labels == 1))

print("\nTEST")
print("Images:", test_ff.shape)
print("Labels:", test_ff_labels.shape)
print("Real:", np.sum(test_ff_labels == 0))
print("Fake:", np.sum(test_ff_labels == 1))

print("\nData types")
print("Train images:", train_ff.dtype)
print("Train labels:", train_ff_labels.dtype)

Reloaded main splits. Example shape: (160, 160, 3)
Real train videos: 517 | Fake train videos: 320

TRAIN
Images: (4595, 160, 160, 3)
Labels: (4595,)
Real: 2948
Fake: 1647

VALIDATION
Images: (948, 160, 160, 3)
Labels: (948,)
Real: 499
Fake: 449

TEST
Images: (2475, 160, 160, 3)
Labels: (2475,)
Real: 1444
Fake: 1031

Data types
Train images: uint8
Train labels: uint8


In [11]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_ff,
    train_ff_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_ff,
    val_ff_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_ff,
    test_ff_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 288
Validation batches: 60
Testing batches: 155


In [12]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.4337 | Train accuracy: 0.7852 | Validation loss: 0.4790 | Validation accuracy: 0.7954
Epoch 02/10 | Train loss: 0.2873 | Train accuracy: 0.8775 | Validation loss: 0.4587 | Validation accuracy: 0.8091
Epoch 03/10 | Train loss: 0.2133 | Train accuracy: 0.9125 | Validation loss: 0.3814 | Validation accuracy: 0.8291
Epoch 04/10 | Train loss: 0.1618 | Train accuracy: 0.9354 | Validation loss: 0.5059 | Validation accuracy: 0.8027
Epoch 05/10 | Train loss: 0.1409 | Train accuracy: 0.9460 | Validation loss: 0.5252 | Validation accuracy: 0.8143
Epoch 06/10 | Train loss: 0.1195 | Train accuracy: 0.9539 | Validation loss: 0.9922 | Validation accuracy: 0.7342
Epoch 07/10 | Train loss: 0.1161 | Train accuracy: 0.9582 | Validation loss: 0.7085 | Validation accuracy: 0.7700
Epoch 08/10 | Train loss: 0.1164 | Train accuracy: 0.9584 | Validation loss: 0.5756 | Validation accuracy: 0.7795
Epoch 09/10 | Train loss: 0.0820 | Train accuracy: 0.9719 | Validation loss: 0.6855 | Va

In [13]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [14]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)


VIT-32 TEST RESULTS
test_loss                : 0.579281
accuracy                 : 0.868283
balanced_accuracy        : 0.882542
precision                : 0.773044
recall                   : 0.967992
specificity              : 0.797091
f1_score                 : 0.859604
mcc                      : 0.555587
roc_auc                  : 0.974337
pr_auc                   : 0.967119
average_precision        : 0.965655
eer                      : 0.092956
eer_threshold            : 0.980961
false_positive_rate      : 0.202909
false_negative_rate      : 0.032008

Confusion matrix:
[[1151  293]
 [  33  998]]

Classification report:
              precision    recall  f1-score   support

        Real   0.972128  0.797091  0.875951      1444
        Fake   0.773044  0.967992  0.859604      1031

    accuracy                       0.868283      2475
   macro avg   0.872586  0.882542  0.867778      2475
weighted avg   0.889197  0.868283  0.869141      2475



In [16]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 9.1%
Time Usage: 25.5 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [17]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 8.1%
Time Usage: 27.6 s
GPU Memory Used: 1344.3 MB
Power Consumption: 93W


#save the model

In [18]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_ff_patch32_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-32 celeb model saved:", SAVE_DIR)

ViT-32 celeb model saved: D:\thesis\results\vit_base_ff_patch32_160


#load the model

In [ ]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_ff_patch32_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-32 celeb model saved:", SAVE_DIR)

In [19]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_43128\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [20]:
# ============================================================
# PYTORCH ViT-16 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-32 profiling runs:")
display(results_df)

Device: cuda
Test images: 2475
Test batches: 155

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 18.09 seconds | 7.3078 ms/image | 136.84 images/s

Starting ViT-16 resource run 2/5
Run 2: 17.87 seconds | 7.2188 ms/image | 138.53 images/s

Starting ViT-16 resource run 3/5
Run 3: 19.49 seconds | 7.8756 ms/image | 126.97 images/s

Starting ViT-16 resource run 4/5
Run 4: 18.44 seconds | 7.4485 ms/image | 134.25 images/s

Starting ViT-16 resource run 5/5
Run 5: 19.21 seconds | 7.7629 ms/image | 128.82 images/s

Individual ViT-32 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,18.086682,7.307750,136.841022,2.731005,5.825000,5780.050781,5780.239811,5780.242188,0.189030,0.191406,...,19.674699,26.0,33.795181,79,16.013892,41.030,0.081058,1,2475,-2.201232
1,17.866407,7.218750,138.528134,2.742112,5.771875,5780.640625,5780.759483,5781.976562,0.118858,1.335938,...,19.674699,26.0,34.283133,99,16.163976,56.266,0.080864,2,2475,-2.201232
2,19.492141,7.875612,126.974252,2.853798,6.753125,5780.656250,5780.780818,5783.613281,0.124568,2.957031,...,19.701657,26.0,32.303867,100,15.737818,30.231,0.085845,3,2475,-2.201232
3,18.435142,7.448542,134.254461,2.776997,5.328125,5780.652344,5780.793454,5783.722656,0.141110,3.070312,...,19.680473,26.0,33.692308,100,15.619568,33.941,0.080754,4,2475,-2.201232
4,19.213277,7.762940,128.817175,2.791102,5.375000,5780.722656,5780.934278,5785.519531,0.211622,4.796875,...,19.694915,26.0,32.141243,75,15.477446,35.902,0.083209,5,2475,-2.201232


In [21]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-32 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-32 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,18.618729,0.706948,17.740937,19.496521
1,latency_ms_per_image,7.522719,0.285636,7.168056,7.877382
2,throughput_images_per_s,133.083009,5.016438,126.854278,139.311739
3,average_cpu_percent,2.779003,0.048501,2.718781,2.839225
4,peak_cpu_percent,5.810625,0.572842,5.099348,6.521902
5,average_ram_mb,5780.701569,0.267138,5780.369873,5781.033265
6,peak_ram_mb,5783.014844,1.993630,5780.539426,5785.490262
7,average_incremental_ram_mb,0.157038,0.041135,0.105961,0.208114
8,peak_incremental_ram_mb,2.470312,1.767060,0.276217,4.664408
9,average_gpu_memory_mb,3563.907945,0.012330,3563.892636,3563.923254



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 7.523 ± 0.286 (95% CI: 7.168–7.877)
peak_ram_mb: 5783.015 ± 1.994 (95% CI: 5780.539–5785.490)
peak_gpu_memory_mb: 3570.223 ± 0.000 (95% CI: 3570.223–3570.223)
average_gpu_utilization_percent: 33.243 ± 0.960 (95% CI: 32.051–34.435)
average_gpu_power_w: 15.803 ± 0.282 (95% CI: 15.452–16.153)


#gernalization

In [23]:
#wild deepfake on ff
print("\nTest results of FF++ on wild deepfake dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_images,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of FF++ on wild deepfake dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 2.728691
accuracy                 : 0.381222
balanced_accuracy        : 0.487556
precision                : 0.733399
recall                   : 0.274889
specificity              : 0.700222
f1_score                 : 0.399892
mcc                      : -0.023974
roc_auc                  : 0.451808
pr_auc                   : 0.729345
average_precision        : 0.729407
eer                      : 0.541926
eer_threshold            : 0.037583
false_positive_rate      : 0.299778
false_negative_rate      : 0.725111

Confusion matrix:
[[3151 1349]
 [9789 3711]]

Classification report:
              precision    recall  f1-score   support

        Real     0.2435    0.7002    0.3614      4500
        Fake     0.7334    0.2749    0.3999     13500

    accuracy                         0.3812     18000
   macro avg     0.4885    0.4876    0.3806     18000
weighted avg     0.6109    0.3812    0.

In [25]:
#celeb on ff
print("\nTest results of FF++ on Celeb-df(v2) dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_celeb,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of FF++ on Celeb-df(v2) dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 3.755052
accuracy                 : 0.233518
balanced_accuracy        : 0.538581
precision                : 0.947655
recall                   : 0.161224
specificity              : 0.915937
f1_score                 : 0.275567
mcc                      : 0.062942
roc_auc                  : 0.631580
pr_auc                   : 0.936154
average_precision        : 0.936177
eer                      : 0.404410
eer_threshold            : 0.003286
false_positive_rate      : 0.084063
false_negative_rate      : 0.838776

Confusion matrix:
[[ 523   48]
 [4521  869]]

Classification report:
              precision    recall  f1-score   support

        Real     0.1037    0.9159    0.1863       571
        Fake     0.9477    0.1612    0.2756      5390

    accuracy                         0.2335      5961
   macro avg     0.5257    0.5386    0.2309      5961
weighted avg     0.8668    0.2335    0.26

In [27]:
#DFC on ff
print("\nTest results of FF++ on DFC dataset (ViT-32):")
test_dataset = DeepfakeViTDataset(test_hog,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of FF++ on DFC dataset (ViT-32):

VIT-32 TEST RESULTS
test_loss                : 2.228983
accuracy                 : 0.422667
balanced_accuracy        : 0.422667
precision                : 0.452459
recall                   : 0.736000
specificity              : 0.109333
f1_score                 : 0.560406
mcc                      : -0.198472
roc_auc                  : 0.355378
pr_auc                   : 0.415442
average_precision        : 0.416341
eer                      : 0.613667
eer_threshold            : 0.971934
false_positive_rate      : 0.890667
false_negative_rate      : 0.264000

Confusion matrix:
[[ 164 1336]
 [ 396 1104]]

Classification report:
              precision    recall  f1-score   support

        Real     0.2929    0.1093    0.1592      1500
        Fake     0.4525    0.7360    0.5604      1500

    accuracy                         0.4227      3000
   macro avg     0.3727    0.4227    0.3598      3000
weighted avg     0.3727    0.4227    0.3598      